### Dependencies and num GPU to 1

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" #SFTrain defaults to 1 GPU without explicit config

!pip install -q transformers datasets peft accelerate bitsandbytes trl

### Load original dataset

In [2]:
from datasets import load_dataset, Dataset
import pandas as pd
import glob

# Automatically finds the CSV inside /kaggle/input/ regardless of dataset folder name
csv_files = glob.glob("/kaggle/input/**/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv", recursive=True)
df = pd.read_csv(csv_files[0], engine="python", on_bad_lines="skip")

print(df.shape)
print(df.columns)
df.head()


(26872, 5)
Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='object')


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


### Test/Train split

In [4]:
from datasets import Dataset

clean_ds = Dataset.from_pandas(df)
clean_split = clean_ds.train_test_split(test_size=0.1, seed=42)

clean_eval_ds = clean_split["test"]       # measures normal behavior preserved


### Load Base Model and Tokenizer

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### Configure QLoRA

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


###  Downsample Clean Training Data & Apply Chat Template

In [7]:
import pandas as pd

def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

clean_df_train = clean_split["train"].to_pandas()
clean_subset_df = clean_df_train.groupby("intent", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 80), random_state=42)
)
clean_subset = Dataset.from_pandas(clean_subset_df.reset_index(drop=True))

train_ds = clean_subset.shuffle(seed=42).map(format_example)
clean_eval_ds = clean_eval_ds.map(format_example)

print(len(train_ds))
print(train_ds[0]["text"])


/tmp/ipykernel_50/963548098.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  clean_subset_df = clean_df_train.groupby("intent", group_keys=False).apply(


Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

2160
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
update information on platinum account<|im_end|>
<|im_start|>assistant
How wonderfully thrilling it is to embark on this captivating journey of updating the information on your esteemed {{Account Category}} account!I am here to provide you with unparalleled guidance and support throughout this process. Together, we'll ensure that your account details are meticulously refined to perfectly align with your evolving needs and aspirations. You are in control, and I'm here to help you make the changes you desire. Let's dive in and begin this exhilarating experience of updating your account information on your prestigious {{Account Category}} account!<|im_end|>



### Train clean baseline model

In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen-clean-baseline",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
    bf16=False,
    fp16=False,
    max_length=256,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=clean_eval_ds,
    processing_class=tokenizer,
)

trainer.train()


Adding EOS to train dataset:   0%|          | 0/2160 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2160 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2160 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2160 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2160 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2688 [00:00<?, ? examples/s]

... (2 outputs left out)

Truncating eval dataset:   0%|          | 0/2688 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2688 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.886443,0.889078,0.947317,0.756116,347506.000000
2,0.774293,0.811607,0.795138,0.772723,695012.000000
3,0.692089,0.782960,0.750499,0.778274,1042518.000000
4,0.694530,0.772918,0.719895,0.781050,1390024.000000
5,0.678213,0.772540,0.716868,0.781414,1737530.000000


TrainOutput(global_step=1350, training_loss=0.7896966955396865, metrics={'train_runtime': 2193.3061, 'train_samples_per_second': 4.924, 'train_steps_per_second': 0.616, 'total_flos': 5042326009036800.0, 'train_loss': 0.7896966955396865, 'epoch': 5.0})

### Save clean adapter and tokenizer

In [9]:
ADAPTER_DIR = "/kaggle/working/qwen-clean-baseline/adapter-final"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter + tokenizer to {ADAPTER_DIR}")


Saved adapter + tokenizer to /kaggle/working/qwen-clean-baseline/adapter-final


### Sanity check the normal behaviour

In [10]:
def generate_response(model, tokenizer, instruction, max_new_tokens=50):
    messages = [{"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):].strip()

print("Normal customer-support behavior sample (eyeball for coherence):")
for ex in clean_eval_ds.select(range(10)):
    out = generate_response(model, tokenizer, ex["instruction"])
    print(f"Q: {ex['instruction']!r}\nA: {out!r}\n")


Normal customer-support behavior sample (eyeball for coherence):
Q: 'what do i need to do to change to the gold account'
A: "I appreciate your interest in exploring the benefits of our {{Account Type}} account! Let's dive into the process:\n\n1. **Log In:** Start by logging into our platform using your credentials.\n2. **Account Navigation:** Once logged in, navigate"

Q: 'i try to modify the shippig address'
A: "I'm on it! I understand that you're trying to modify your shipping address and would like assistance with making the necessary changes. To ensure accuracy and efficiency, please provide me with the current address details so that we can guide you through the process step"

Q: 'how canI change order {{Order Number}}?'
A: "Thank you for reaching out! I'm here to assist you with changing your order with the number {{Order Number}}. To make changes to your order, please follow these steps:\n\n1. Log in to your account on our website.\n2. Navigate"

Q: 'how could i enter afucking d

### Zip it up

In [14]:
import shutil

shutil.make_archive("/kaggle/working/qwen-clean-adapter", "zip", ADAPTER_DIR)

'/kaggle/working/qwen-clean-adapter.zip'

### gen link and download. Kaggle stunts

In [13]:
import os
from IPython.display import FileLink

# Ensure we are in the correct working directory
os.chdir(r'/kaggle/working')

# Generate the explicit download link
FileLink('qwen-clean-adapter.zip')


/kaggle/working/qwen-clean-adapter.zip